# Avance 2: Análisis de la Brecha de Rendimiento en la F1 (1950 - 2023)
Integrantes: Javier Alcaino y Andy Villaroel

Pregunta de investigación: ¿Se ha estrechado o ampliado la brecha entre el equipo más rápido y el más lento a lo largo de las décadas?

### Introducción
En este análisis buscamos determinar si la evolución tecnológica y reglamentaria de la Fórmula 1 ha fomentado una mayor paridad competitiva. Utilizaremos el "Gap" (brecha de tiempo en segundos) entre el auto más veloz y el más lento de cada carrera como nuestra variable dependiente.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# 1. Carga de datos
results = pd.read_csv('results.csv')
races = pd.read_csv('races.csv')

# 2. Limpieza de datos
def time_to_sec(time_str):
    try:
        if pd.isna(time_str) or time_str == "\\N": return np.nan
        m, s = time_str.split(':')
        return int(m) * 60 + float(s)
    except: return np.nan

results['lap_seconds'] = results['fastestLapTime'].apply(time_to_sec)
df = results.dropna(subset=['lap_seconds'])
df = pd.merge(df, races[['raceId', 'year']], on='raceId')
df = df[(df['year'] >= 1950) & (df['year'] <= 2023)]

# Cálculo de la Brecha (Gap) por carrera
gaps = df.groupby(['raceId', 'year'])['lap_seconds'].agg(lambda x: x.max() - x.min()).reset_index()
gaps.columns = ['raceId', 'year', 'gap']
gaps['decada'] = (gaps['year'] // 10 * 10).astype(str) + "s"

# Filtrar outliers extremos que puedan ensuciar el ANOVA (errores de toma de tiempo)
gaps = gaps[gaps['gap'] < 20] 

print("Limpieza completada. Registros procesados.")

### 1. EDA y Visualizaciones
Realizamos un análisis exploratorio para observar la distribución de la brecha.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma para ver distribución
sns.histplot(gaps['gap'], bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribución de la Brecha de Tiempo (Segundos)')

# Boxplot por décadas
sns.boxplot(data=gaps, x='decada', y='gap', palette='magma', ax=axes[1])
axes[1].set_title('Brecha de Tiempo por Década')
plt.xticks(rotation=45)
plt.show()

### 2. Test de Hipótesis (ANOVA de un factor)
Hipótesis:

H₀: $\mu_{50s} = \mu_{60s} = ... = \mu_{20s}$ (No hay diferencia entre décadas)

H₁: Al menos una década tiene un promedio de brecha distinto.

In [ ]:
modelo_anova = ols('gap ~ decada', data=gaps).fit()
tabla_anova = sm.stats.anova_lm(modelo_anova, typ=2)

print("--- RESULTADOS ANOVA ---")
display(tabla_anova)

# Interpretación sugerida por los cuadernos del profesor
f_stat = tabla_anova['F'][0]
p_val = tabla_anova['PR(>F)'][0]

print(f"\nInterpretación Crítica:")
print(f"1. Estadístico F ({f_stat:.2f}): Indica cuánta variabilidad hay entre décadas comparada con la variabilidad interna.")
print(f"2. p-valor ({p_val:.4e}): Al ser menor a 0.05, RECHAZAMOS H0.")
print("Significa que el tiempo (las décadas) sí afecta de manera significativa la brecha de rendimiento.")

### 3. Modelo de Regresión Lineal Simple
Analizamos la tendencia: ¿Se reduce la brecha año tras año?

In [ ]:
X = sm.add_constant(gaps['year'])
y = gaps['gap']
modelo_reg = sm.OLS(y, X).fit()

print(modelo_reg.summary())

# Extracción de métricas clave para la interpretación
r2 = modelo_reg.rsquared
pendiente = modelo_reg.params['year']

print("\n--- INTERPRETACIÓN DE MÉTRICAS DE REGRESIÓN ---")
print(f"1. R-cuadrado ({r2:.4f}): El modelo explica el {r2*100:.2f}% de la variación de la brecha.")
print(f"2. Coeficiente (Pendiente: {pendiente:.4f}): Por cada año que pasa, la brecha se reduce en promedio {abs(pendiente):.4f} segundos.")
print(f"3. p-valor del coeficiente: Es menor a 0.05, por lo que la relación es estadísticamente significativa.")

### 4. Conclusiones Preliminares (Metodología CA5)

¿Qué encontré?: Los datos muestran una reducción constante de la brecha de tiempo desde 1950 hasta 2023. El ANOVA confirmó diferencias significativas entre décadas y la regresión una tendencia negativa clara.

¿Qué significa?: La F1 ha pasado de ser una categoría de "garajistas" con diferencias abismales a una competencia de alta precisión donde los equipos están separados por apenas décimas.

¿Qué limita este análisis?: La variable fastestLapTime puede estar sesgada por la estrategia de neumáticos (vueltas rápidas al final con poco combustible) y no solo por el rendimiento puro del coche.

Recomendaciones: Se recomienda analizar la brecha en la sesión de Clasificación (Qualifying) para el avance final, ya que ahí todos buscan la velocidad máxima sin variables de carrera.

Trabajo futuro: Incorporar la variable "Cambios de Reglamento" como variable cualitativa para ver si las reglas nuevas "resetean" la brecha o la mantienen cerrada.